In [1]:
# Import necessary libraries[cite: 2]
import os
import random
import time
import matplotlib.pyplot as plt
import ollama
import pandas as pd
from tqdm import tqdm
from wordcloud import WordCloud



In [ ]:
# Path of the  data to be labeled(input_data_path)
input_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Exploratory_Data_Analysis/final_bitcoin_data.csv"
# Path where the results to be saved
output_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Pragmantic_Function_Identification/outputs.csv"

In [ ]:
# Define target column name
target_column = "deepseek_temp0"
# read the input data to a pandas dataframe and check the information about its contents 
input_data = pd.read_csv(input_data_path)
input_data.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           906 non-null    int64 
 1   comment         906 non-null    object
 2   author          906 non-null    object
 3   subreddit       906 non-null    object
 4   score           906 non-null    int64 
 5   ups             906 non-null    int64 
 6   downs           906 non-null    int64 
 7   date            906 non-null    object
 8   created_utc     906 non-null    object
 9   parent_comment  906 non-null    object
dtypes: int64(4), object(6)
memory usage: 70.9+ KB


In [ ]:
# Check if output file exists, otherwise create a new one from input_data
if os.path.exists(output_data_path):
    print("Found an existing file with the same name, continuing labeling...")
    new_labeled_df = pd.read_csv(output_data_path)

    if target_column not in new_labeled_df.columns:
        new_labeled_df[target_column] = pd.NA
else:
    print(
        "File not found with the given name, creating a new file and starting labeling from the beginning"
    )
    new_labeled_df = input_data.copy()
    new_labeled_df[target_column] = pd.NA

# Function to find the starting index for unlabelled rows
def index_finder(df):
    for index, value in df[target_column].items():
        if pd.isna(value) or str(value).strip() == "":
            return index
    return len(df)


start_index = index_finder(new_labeled_df)

if start_index >= len(new_labeled_df):
    print(
        f"\nThe dataset is already fully labeled for column '{target_column}'. No further labeling is required."
    )
else:
    print(
        f"\nContinuing labelling from row index: {start_index} out of {len(new_labeled_df)}"
    )



Found an existing file with the same name, continuing labeling...

Continuing labelling from row index: 691 out of 906


In [ ]:
# Function to extract the pragmatic function using Ollama
def extract_pragmatic_function(parent_comment, response_comment):
    prompt = f"""
    Parent Comment: "{parent_comment}"
    Response Comment: "{response_comment}"

    by reading the paret comment and response comment find out the pragmatic function conveyed in response comment
    Your response should strictly a one word 
    Do not add any punctuation, explanation, or extra text, it should only contain letters 
    """

    try:
        result = ollama.chat(
            model='deepseek-r1:8b',
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a pragmatic function classifier. Respond with"
                        " EXACTLY one word describing the pragmatic function"
                        " behind the comment."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            options={
                "temperature": 0.0,
            },
        )
        original_answer = result["message"]["content"].strip()

        if original_answer:
            return (
                original_answer.split()[0]
                .replace(".", "")
                .replace(",", "")
                .lower()
            )
        else:
            return "neutral"
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")




In [ ]:
# Run inference and incrementally save progress
if start_index < len(new_labeled_df):
    print("\nIterating through remaining unlabelled rows via local Ollama instance...")
    start_time = time.time()

    non_labelled_rows = new_labeled_df.iloc[start_index:]
    pbar = tqdm(
        non_labelled_rows.iterrows(),
        total=len(non_labelled_rows),
        miniters=25,
    )

    for index, row in pbar:
        try:
            prediction = extract_pragmatic_function(
                row["parent_comment"], row["comment"]
            )
            new_labeled_df.at[index, target_column] = prediction

            if index % 25 == 0:
                pbar.set_description(f"Processing Row {index}")

        except Exception as e:
            tqdm.write(f"Row {index} skipped due to error: {e}")
            continue

        # Save checkpoint every 10 rows
        if index % 10 == 0:
            new_labeled_df.to_csv(output_data_path, index=False)

    # Final save and close progress bar
    pbar.close()
    new_labeled_df.to_csv(output_data_path, index=False)

    # Calculate and display total elapsed time
    end_time = time.time()
    total_seconds = int(end_time - start_time)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    print(f"\nDataset labeling complete!")
    print(f"Total rows processed in this run: {len(non_labelled_rows)}")
    print(f"Total time elapsed:   {hours}h {minutes:02d}m {seconds:02d}s")




Iterating through remaining unlabelled rows via local Ollama instance...


  0%|          | 0/215 [00:18<?, ?it/s]


KeyboardInterrupt: 